<a href="https://colab.research.google.com/github/BenayaL/Cloud_Computing_Project/blob/main/Tirgul1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive')

Mounted at /content/drive
Lidor Ben-David lidor.ben.david@e.braude.ac.il cloud web physics automation probability https://www.youtube.com/watch?v=dQw4w9WgXcQ&list=RDdQw4w9WgXcQ&start_radio=1


In [2]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Load and prepare the data (Using our clean reading method)
file_path = '/content/drive/My Drive/students.txt'
column_headers = ['First Name', 'Last Name', 'Email', 'Courses Taken', 'Interesting Link']

try:
    df = pd.read_csv(file_path, sep=r'\s*\|\s*', header=None, engine='python')
    df = df.iloc[:, :5] # Keep only the first 5 columns
    df.columns = column_headers
    df.dropna(how='all', inplace=True)
except FileNotFoundError:
    print(f"Error: Could not find the file at {file_path}")
    # Creating dummy data if file is missing
    df = pd.DataFrame([["Uriel", "Pekelis", "uriel@email.com", "Cloud", "link"]], columns=column_headers)

# Create a "Full Name" column to make the dropdown look nice
df['Full Name'] = df['First Name'] + " " + df['Last Name']

# Add a "Favorite Program" column if we haven't already
if 'Favorite Program' not in df.columns:
    df['Favorite Program'] = "Not assigned yet"

# 2. Create the Widgets
# The dropdown uses the 'Full Name' column for its options
student_dropdown = widgets.Dropdown(
    options=['-- Select a Student --'] + df['Full Name'].tolist(),
    description='Student:',
    disabled=False,
)

info_output = widgets.Output() # Area to show the student's static info
status_output = widgets.Output() # Area to show success messages

# Editing widgets (Disabled by default until a student is selected)
program_input = widgets.Text(description='Fav Program:', disabled=True)
save_button = widgets.Button(description='Save Edit', button_style='success', disabled=True)

# 3. Define the interaction functions
def on_dropdown_change(change):
    """This runs every time a new name is picked from the dropdown."""
    if change['type'] == 'change' and change['name'] == 'value':
        selected_name = change['new']

        info_output.clear_output()
        status_output.clear_output()

        # If they go back to the default prompt, disable everything
        if selected_name == '-- Select a Student --':
            program_input.value = ''
            program_input.disabled = True
            save_button.disabled = True
            return

        # Enable the editing tools
        program_input.disabled = False
        save_button.disabled = False

        # Find the specific student's row in the DataFrame
        # .iloc[0] grabs the first matching row as a dictionary-like object
        student_data = df[df['Full Name'] == selected_name].iloc[0]

        # Display their information
        with info_output:
            print(f"--- Information for {selected_name} ---")
            print(f"Email:   {student_data['Email']}")
            print(f"Courses: {student_data['Courses Taken']}")
            print(f"Link:    {student_data['Interesting Link']}")

        # Pre-fill the input box with whatever their current favorite program is
        program_input.value = student_data['Favorite Program']

def on_save_clicked(b):
    """This runs when the Save button is clicked."""
    selected_name = student_dropdown.value
    new_program = program_input.value

    # 1. Update the DataFrame in memory
    row_index = df[df['Full Name'] == selected_name].index[0]
    df.at[row_index, 'Favorite Program'] = new_program

    # 2. Save the updated DataFrame back to the text file
    try:
        # We drop 'Full Name' before saving so we don't change the original file structure
        cols_to_save = [c for c in df.columns if c != 'Full Name']

        df[cols_to_save].to_csv(
            file_path,
            sep='|',
            index=False,
            header=False
        )

        with status_output:
            clear_output()
            print(f"✅ Saved to file! Updated {selected_name}'s program to: '{new_program}'")

    except Exception as e:
        with status_output:
            clear_output()
            print(f"❌ Error saving to file: {e}")

# 4. Connect the functions to the widgets
student_dropdown.observe(on_dropdown_change)
save_button.on_click(on_save_clicked)

# 5. Display everything in a neat layout
# HBox puts things side-by-side, VBox stacks them vertically
edit_row = widgets.HBox([program_input, save_button])
ui_layout = widgets.VBox([student_dropdown, info_output, edit_row, status_output])

display(ui_layout)

To fix the GitHub rendering error caused by `metadata.widgets`, you can run the following code. This code will remove the `widgets` metadata from your current notebook, which often resolves the issue. You should run this **before saving and uploading your notebook to GitHub**.

In [8]:
import nbformat
from IPython import get_ipython
import os

# Get the current notebook path using the connection file
# This method is more robust in Colab than trying to infer from sys.argv
try:
    # Get the connection file path
    connection_file = get_ipython().config['IPKernelApp']['connection_file']
    # Extract the notebook path from the connection file path
    notebook_filename = os.path.basename(connection_file).replace('kernel-', '').replace('.json', '.ipynb')
    # Check if a notebook with this filename exists in common Colab locations
    possible_paths = [
        notebook_filename, # Current directory
        os.path.join('Colab Notebooks', notebook_filename), # Common Colab folder
        os.path.join(os.path.expanduser('~'), 'drive', 'MyDrive', notebook_filename), # Direct MyDrive path if not using chdir
        os.path.join(os.path.expanduser('~'), 'drive', 'MyDrive', 'Colab Notebooks', notebook_filename) # Full path to common Colab folder
    ]
    resolved_path = None
    for path_attempt in possible_paths:
        if os.path.exists(path_attempt):
            resolved_path = path_attempt
            break

    if resolved_path:
        notebook_path = resolved_path
    else:
        # Fallback if automatic detection fails: assume a specific known path if 'Tirgul1.ipynb'
        # This handles the case where the user might have named their notebook 'Tirgul1.ipynb'
        # and it's located in 'Colab Notebooks'
        if notebook_filename == 'Tirgul1.ipynb' and os.path.exists('/content/drive/MyDrive/Colab Notebooks/Tirgul1.ipynb'):
            notebook_path = '/content/drive/MyDrive/Colab Notebooks/Tirgul1.ipynb'
        else:
            raise FileNotFoundError(f"Could not find an .ipynb file named '{notebook_filename}' or similar in common locations.")

except Exception as e:
    print(f"Warning: Automatic notebook path detection failed: {e}. Attempting a specific fallback.")
    # Explicitly setting the path for 'Tirgul1.ipynb' as observed in the file system
    notebook_path = "/content/drive/MyDrive/Colab Notebooks/Tirgul1.ipynb"
    if not os.path.exists(notebook_path):
        print(f"Error: Fallback path '{notebook_path}' also not found.")
        notebook_path = None # Set to None if even fallback fails

if notebook_path:
    print(f"Attempting to clean metadata for: {notebook_path}")
    # Read the notebook
    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook = nbformat.read(f, as_version=4)

        # Remove 'widgets' from metadata. If it exists.
        if 'metadata' in notebook and 'widgets' in notebook['metadata']:
            del notebook['metadata']['widgets']
            print(f"Removed 'widgets' metadata from {notebook_path}")
        else:
            print(f"'widgets' metadata not found or already removed in {notebook_path}")

        # Write the cleaned notebook back to the same path
        with open(notebook_path, 'w', encoding='utf-8') as f:
            nbformat.write(notebook, f)

        print("Notebook metadata cleaned. Remember to download and then upload this notebook to GitHub.")
    except FileNotFoundError:
        print(f"Error: Notebook file '{notebook_path}' not found. Please ensure it exists or verify the path.")
    except Exception as e:
        print(f"An unexpected error occurred during cleaning: {e}")
else:
    print("Notebook path could not be determined. Please manually set 'notebook_path' and rerun the cell if the above failed.")

Attempting to clean metadata for: /content/drive/MyDrive/Colab Notebooks/Tirgul1.ipynb
Removed 'widgets' metadata from /content/drive/MyDrive/Colab Notebooks/Tirgul1.ipynb
Notebook metadata cleaned. Remember to download and then upload this notebook to GitHub.
